# Pipeline d'Analyse Exploratoire — Boston House Dataset



Ce script réalise une **analyse exploratoire des données (EDA)** sur le dataset Boston House. Il est structuré en plusieurs fonctions, chacune couvrant un aspect de l'analyse.

---

### `analyse_missing_values(df)`

Affiche **3 visualisations** des valeurs manquantes côte à côte :

- **Matrice** : montre où se trouvent les valeurs manquantes dans le dataset
- **Barres** : nombre de valeurs présentes par colonne
- **Heatmap** : corrélation entre les colonnes ayant des valeurs manquantes


### `analyse_distributions(df)`

Trace un **histogramme avec courbe KDE** pour chaque colonne numérique du dataset.

- Permet de visualiser la forme de la distribution (normale, asymétrique, bimodale...)
- Une sous-figure est créée par colonne numérique


### `analyse_correlations(df)`

Affiche une **heatmap de corrélation** entre toutes les variables numériques.

- Les valeurs vont de **-1** (corrélation négative forte) à **+1** (corrélation positive forte)


---

### `analyse_boxplots(df)`

Affiche des **boxplots** pour toutes les colonnes numériques sur un même graphique.

- Permet de détecter les **outliers** (valeurs aberrantes)
- Montre la médiane, les quartiles et l'étendue des données


---

### `analyse_categorical_distributions(df, to_drop=None)`

Affiche des **countplots** pour toutes les colonnes catégorielles (`object`, `category`, `bool`).

- Le paramètre `to_drop` permet d'exclure certaines colonnes avant l'analyse
- Les graphiques sont organisés en grille de **3 colonnes**

---


## Analyse – Premières impressions

À première vue, on remarque de nombreuses corrélations non linéaire entre différentes variables. Cela n’est pas étonnant, car il s’agit d’analyses portant sur des données démographiques.

Par exemple, on peut observer une corrélation entre le pourcentage de surface industrielle et le taux d’oxyde d’azote.

On constate également une distribution irrégulière concernant le taux de professeurs par élève. Cela soulève une question sur le découpage des zones : est-il possible que ce découpage amplifie certains taux, notamment si plusieurs écoles sont regroupées dans une même zone ?

On observe une forte concentration de population noire dans certaines zones, ce qui reflète une répartition démographique inégale.

De plus, on remarque de nombreux outliers (valeurs aberrantes) dans le taux de criminalité. Cela peut s’expliquer par le fait que la majorité des individus ne sont pas criminels, et que les cas où ils le sont sortent souvent de la norme.

On observe également beaucoup d’outliers concernant le pourcentage de population noire, ce qui reflète ici la disparité démographique entre les zones.

Il en va de même pour les grandes surfaces de terrains résidentiels, qui traduisent des disparités économiques importantes.

## Conclusion

Je ne vois pas de données susceptibles de biaiser la représentation réelle des disparités socio-économiques du terrain, je choisis donc de conserver l’ensemble des variables.

Les outliers observés ne semblent pas être des erreurs de mesure mais plutôt des valeurs réelles reflétant des situations réelles. Ils apportent donc une information pertinente et doivent être pris en compte dans le modèle.

⚠️ Une vigilance éthique reste toutefois nécessaire concernant l’utilisation de statistiques ethniques, même si celles-ci sont autorisées aux États-Unis.



In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import missingno as msno

from analyse import analyse_boxplots, analyse_categorical_distributions, analyse_correlations, analyse_distributions, analyse_missing_values
    
df = pd.read_csv("./data_set_boston_house.csv")
analyse_missing_values(df)
analyse_distributions(df)
analyse_correlations(df)
analyse_boxplots(df)
analyse_categorical_distributions(df, to_drop=None)



## Prétraitement des données

Le dataset ne contient **aucune valeur manquante** et uniquement des **variables numériques**.

Un traitement minimal suffirait ici, mais le pipeline est conçu pour gérer automatiquement l'arrivée de nouvelles données qui pourraient enrichir le dataset (variables catégorielles, valeurs manquantes).

Aucun outlier n'a été retiré.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

from preprocess import preprocessingTechnique


df = pd.read_csv("./data_set_boston_house.csv")
# Utilisation
X_processed, y, preprocessor, df_final = preprocessingTechnique(
    df,
    target_col="valeur_mediane_logements"
)
df_final.head()





## Entrainement des données 

**Gradient Boosting gagne clairement :**
- RMSE de 2.492 → erreur moyenne de **±2 492$** sur le prix
- R² de 0.915 → explique **91.5%** des variations de prix

**Gros écart entre modèles linéaires et ensemblistes :**
- Les modèles ensemblistes (Gradient Boosting, Random Forest) sont **2x plus précis** que les modèles linéaires
- Cela confirme que la relation entre les features et le prix n'est pas linéaire



In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from tensorflow.keras.callbacks import EarlyStopping
import pandas as pd
from nn_model import compare_models


from preprocess import preprocessingTechnique,split


df = pd.read_csv("./data_set_boston_house.csv")
df = df.drop_duplicates()

# Utilisation
X_processed, y, preprocessor, df_final = preprocessingTechnique(
    df,
    target_col="valeur_mediane_logements"
)

X_train, X_test, y_train, y_test = split(X_processed,y)

compare_models(X_train, X_test, y_train, y_test, 50)




## Entrainement des données + cross validation 


La validation simple donnait `2.492` pour Gradient Boosting. La CV donne `4.176`.
le modèle était trop optimiste.

> Lors du split simple, les 20% de test sont tombés sur des données "faciles" par chance.
> La CV révèle la **vraie performance** sur l'ensemble du dataset.


### Les modèles linéaires sont plus honnêtes

| Modèle | Écart | Interprétation |
|---|---|---|
| Ridge | +0.606 | écart faible, comportement stable |
| Linear Regression | +0.900 | peu sensible au split |
| Gradient Boosting | +1.684 | écart énorme, très sensible aux données |
| Random Forest | +1.606 | idem |

Les modèles linéaires ne "mémorisent" pas les données d'entraînement —
leur performance est **cohérente** peu importe le split.
Probable surraprentissage.

---

### Ce que dit le `RMSE ±`

Tous les modèles sont **instables** (± élevé) — signe que le dataset Boston Housing
est **trop petit** (506 lignes) pour avoir des folds vraiment représentatifs.

---

### Conclusion

> La validation simple était **trop optimiste**. La cross-validation révèle que Gradient Boosting
> reste le meilleur modèle, mais avec une erreur réelle de **±4 176$** et non ±2 492$.





In [6]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from tensorflow.keras.callbacks import EarlyStopping
import pandas as pd
from nn_model import compare_cv


from preprocess import preprocessingTechnique,split


df = pd.read_csv("./data_set_boston_house.csv")
df = df.drop_duplicates()
# Utilisation
X_processed, y, preprocessor, df_final = preprocessingTechnique(
    df,
    target_col="valeur_mediane_logements"
)

compare_cv(X_processed, y,5)


        RAPPORT DE PREPROCESSING
  • Colonnes numériques (13) : ['taux_criminalite', 'pct_terrain_residentiel', 'pct_surface_industrie', 'adjacence_charles_river', 'concentration_nox', 'nb_pieces_moyen', 'pct_logements_avant_1940', 'distance_centres_emploi', 'index_acces_autoroutes', 'taux_imposition_fonciere', 'ratio_eleves_enseignant', 'indice_population_noire', 'pct_pop_statut_faible']
  • Colonnes catégorielles (0) : []
  • Numériques → imputation médiane + MinMaxScaler
  • Valeurs manquantes avant : 0 → après : 0
  • Dimensions avant  : (506, 13)
  • Dimensions après  : (506, 13)


2026-04-15 16:28:09.866 | SUCCESS  | nn_model:compare_cv:224 - [CROSS-VALIDATION 5 FOLDS] Run du 2026-04-15_16-28-09
2026-04-15 16:28:09.868 | SUCCESS  | nn_model:compare_cv:225 - Modèle                      RMSE moy   RMSE ±    MAE moy   R² moy
2026-04-15 16:28:09.868 | SUCCESS  | nn_model:compare_cv:226 - -------------------------------------------------------------------
2026-04-15 16:28:09.868 | SUCCESS  | nn_model:compare_cv:228 - Gradient Boosting              4.176    1.105      2.994    0.675
2026-04-15 16:28:09.868 | SUCCESS  | nn_model:compare_cv:228 - Random Forest                  4.418    1.345      3.023    0.627
2026-04-15 16:28:09.868 | SUCCESS  | nn_model:compare_cv:228 - Ridge                          5.465    1.635      3.948    0.459
2026-04-15 16:28:09.868 | SUCCESS  | nn_model:compare_cv:228 - Linear Regression              5.829    1.777      4.250    0.353
2026-04-15 16:28:09.868 | WARNING  | nn_model:compare_cv:231 - Meilleur modèle : Gradient Boosting (RMSE mo

,RMSE moy,RMSE ±,MAE moy,R² moy
Gradient Boosting,4.175528,1.104861,2.994137,0.675000
Random Forest,4.418210,1.345189,3.023213,0.627472
Ridge,5.465045,1.634661,3.947553,0.458858
Linear Regression,5.828659,1.777229,4.249969,0.353276
